# The Normal Distribution
**Topic:** Random Variables & Distributions

In [1]:
import numpy as np
import pandas as pd

import plotly.graph_objects as go
import plotly.express as px

import ipywidgets as widgets
from ipywidgets import interact, interactive, fixed, interact_manual
from ipywidgets import IntSlider, FloatSlider, Dropdown, Button, Output, HBox, VBox, Label

from IPython.display import display, HTML, clear_output
from scipy import stats
from tkh_utils import PALETTE, FONT, base_layout


---
## What you'll explore

By the end of this demo you will be able to:

- **Describe** how the mean μ shifts the center of the bell curve without changing its shape
- **Explain** how the standard deviation σ controls how spread out or concentrated the data is
- **Interpret** the 68–95–99.7 rule by reading areas within 1, 2, and 3 standard deviations

> **Tip:** Start by moving the **σ slider** and watch the curve change shape. Then use the **shade region** dropdown to reveal the 68–95–99.7 rule in action.

---
## How we got here

In *11: Continuous Distributions* we introduced the concept of a PDF and explored the Uniform and Exponential distributions. The Normal distribution is by far the most important member of the continuous family. It emerges whenever many small, independent factors add together, and in *13: Central Limit Theorem*, we will see precisely why that happens so often.

---
## Why this matters for data science

The normal distribution is the assumption baked into linear regression, z-tests, t-tests, ANOVA, and PCA. Feature normalization, residual diagnostics, and anomaly detection all rely on properties of the normal distribution. Even when data is not normally distributed, the models you build often *assume* normality in some component, knowing what that assumption really means makes you a better modeler.

---
## Try it yourself

In [2]:
out = Output()

mu_slider = FloatSlider(
    value=0.0, min=-4.0, max=4.0, step=0.25,
    description="Mean (μ):",
    style={"description_width": "100px"},
    layout=widgets.Layout(width="420px"),
)
sigma_slider = FloatSlider(
    value=1.0, min=0.25, max=3.0, step=0.25,
    description="Std dev (σ):",
    style={"description_width": "100px"},
    layout=widgets.Layout(width="420px"),
)
shade_dropdown = Dropdown(
    options=[
        ("No shading", 0),
        ("Within 1σ  — 68.2% of data", 1),
        ("Within 2σ  — 95.4% of data", 2),
        ("Within 3σ  — 99.7% of data", 3),
    ],
    value=0,
    description="Shade region:",
    style={"description_width": "100px"},
    layout=widgets.Layout(width="420px"),
)

X_MIN, X_MAX = -10, 10
x = np.linspace(X_MIN, X_MAX, 1000)

SHADE_COLORS = {
    1: ("rgba(76,110,245,0.25)",  "rgba(76,110,245,0.7)"),
    2: ("rgba(247,103,  7,0.20)", "rgba(247,103,  7,0.6)"),
    3: ("rgba(47,158, 68,0.15)",  "rgba(47,158, 68,0.5)"),
}

def render(mu, sigma, n_sigma):
    y = stats.norm.pdf(x, mu, sigma)
    y_max = stats.norm.pdf(mu, mu, sigma)

    traces = []

    # Shaded bands — widest first so narrower ones sit on top
    for k in range(n_sigma, 0, -1):
        fill_color, line_color = SHADE_COLORS[k]
        lo, hi = mu - k * sigma, mu + k * sigma
        x_fill = x[(x >= lo) & (x <= hi)]
        y_fill = stats.norm.pdf(x_fill, mu, sigma)
        pct = stats.norm.cdf(hi, mu, sigma) - stats.norm.cdf(lo, mu, sigma)
        traces.append(go.Scatter(
            x=np.concatenate([[lo], x_fill, [hi]]),
            y=np.concatenate([[0],  y_fill,  [0]]),
            fill="toself",
            fillcolor=fill_color,
            line=dict(color=line_color, width=1, dash="dot"),
            name=f"±{k}σ ({pct*100:.1f}%)",
            hoverinfo="skip",
        ))

    traces.append(go.Scatter(
        x=x, y=y,
        mode="lines",
        line=dict(color=PALETTE["primary"], width=3),
        name=f"N(μ={mu:.2f}, σ={sigma:.2f})",
    ))
    traces.append(go.Scatter(
        x=[mu, mu], y=[0, y_max],
        mode="lines",
        line=dict(color=PALETTE["secondary"], width=2, dash="dash"),
        name=f"Mean (μ = {mu:.2f})",
    ))

    layout = base_layout(
        title=f"Normal Distribution — μ = {mu:.2f}, σ = {sigma:.2f}",
        xaxis_title="Value",
        yaxis_title="Probability Density",
    )
    layout.update(
        xaxis=dict(range=[X_MIN, X_MAX]),
        yaxis=dict(range=[0, 0.85]),
        showlegend=True,
    )

    fig = go.Figure(data=traces, layout=layout)
    with out:
        clear_output(wait=True)
        fig.show()

def on_change(change):
    render(mu_slider.value, sigma_slider.value, shade_dropdown.value)

mu_slider.observe(on_change, names="value")
sigma_slider.observe(on_change, names="value")
shade_dropdown.observe(on_change, names="value")

display(VBox([VBox([mu_slider, sigma_slider, shade_dropdown]), out]))
render(mu_slider.value, sigma_slider.value, shade_dropdown.value)

---
## What's happening?

The normal (or Gaussian) distribution describes how values cluster around a central point when many small, independent factors add together, test scores, heights, measurement errors, and countless real-world quantities all follow this pattern.

Two numbers fully define the shape:

| Parameter | Symbol | What it controls |
|-----------|--------|-----------------|
| Mean | μ | Where the peak sits: the center of symmetry |
| Standard deviation | σ | How spread out the data is: larger σ → flatter, wider curve |

$$f(x) = \frac{1}{\sigma\sqrt{2\pi}}\, e^{-\frac{1}{2}\left(\frac{x-\mu}{\sigma}\right)^2}$$

### The 68–95–99.7 rule

No matter what μ and σ are, the same fractions of data always fall within the same number of standard deviations:

- **±1σ → ~68.2% of data**
- **±2σ → ~95.4% of data**
- **±3σ → ~99.7% of data**

Use the shade region dropdown above to see each band fill in, the percentages never change even as you drag μ and σ.

---
## Real-world example: Human height

Adult heights in a population cluster around a mean, with fewer and fewer people toward extremes, exactly the bell-curve shape you explored above. The chart simulates 400 adult heights for men and women separately, using realistic parameters from US population studies.

Notice:

- **Notice:** Each group forms its own bell curve with a different mean, the two distributions overlap, showing that a 170cm height is plausible for both groups
- **Notice:** The spread (σ) is similar for both groups even though the means differ, shape and location are controlled by separate parameters
- **Notice:** The crossover region is where classification is hardest, a 50/50 guess is the best you can do without additional information

> **Discussion question:** If you randomly selected someone who is 185 cm tall, which group are they more likely to come from? How does the overlap region change your certainty?

In [3]:
np.random.seed(42)

# ── Realistic US adult height parameters from population studies (in cm) ───────
groups = {
    "Men":   {"mu": 178, "sigma": 7, "n": 400, "color": PALETTE["primary"]},
    "Women": {"mu": 165, "sigma": 6, "n": 400, "color": PALETTE["secondary"]},
}
x_h = np.linspace(135, 215, 500)
traces = []

for group, p in groups.items():
    heights = np.random.normal(p["mu"], p["sigma"], p["n"])
    traces.append(go.Histogram(
        x=heights, histnorm="probability density", nbinsx=28,
        marker_color=p["color"], opacity=0.45,
        name=f"{group} (n={p['n']})", legendgroup=group,
    ))
    traces.append(go.Scatter(
        x=x_h, y=stats.norm.pdf(x_h, p["mu"], p["sigma"]),
        mode="lines", line=dict(color=p["color"], width=2.5),
        name=f"N(μ={p['mu']}, σ={p['sigma']})",
        legendgroup=group, showlegend=False,
    ))

traces.append(go.Scatter(
    x=[171.5, 171.5], y=[0, 0.068],
    mode="lines", line=dict(color=PALETTE["muted"], width=1.5, dash="dot"),
    name="Approx. crossover (~171.5 cm)",
))

layout = base_layout(
    title="Adult Heights Are Normally Distributed",
    xaxis_title="Height (cm)",
    yaxis_title="Probability Density",
)
layout.update(barmode="overlay")
fig = go.Figure(data=traces, layout=layout)
fig.show()

### Other everyday examples of the normal distribution

| Phenomenon | What μ represents | What σ represents |
|------------|-------------------|-------------------|
| Standardized test scores (SAT, IQ) | Average score in the population | How much scores typically vary |
| Manufacturing tolerances | Target measurement (e.g., bolt diameter) | Precision of the machine |
| Daily stock returns (short windows) | Average daily gain/loss | Volatility of the stock |
| Blood pressure readings | Healthy population average | Natural variation between individuals |
| Measurement error | True value of what's being measured | Accuracy of the instrument |

---
## Key takeaway

> **The mean tells you where the distribution lives; the standard deviation tells you how spread out it is — and no matter those values, 68% of the data always falls within one standard deviation of the mean.**

---
*Next up: The Central Limit Theorem — why the normal distribution appears even when the underlying data is not normal*